# GEO 4편: robots.txt AI 크롤러 분석기

경쟁사 도메인의 AI 크롤러 허용/차단 현황을 한눈에 파악한다.

API 키 없이, 표준 라이브러리만으로 동작한다.

📖 [블로그 원문](https://datanexus-kr.github.io/guides/geo-optimization/004-offsite-geo-strategy/)

## 1. AI 크롤러 목록 정의

2026년 기준, 주요 AI 플랫폼이 사용하는 크롤러 User-Agent 목록이다.

| 크롤러 | 운영사 | 용도 |
|--------|--------|------|
| GPTBot | OpenAI | ChatGPT 학습 데이터 수집 |
| ChatGPT-User | OpenAI | ChatGPT 웹 브라우징 모드 |
| OAI-SearchBot | OpenAI | SearchGPT 실시간 검색 |
| Google-Extended | Google | Gemini AI 학습 |
| PerplexityBot | Perplexity | Perplexity AI 검색 |
| ClaudeBot | Anthropic | Claude 학습 데이터 수집 |
| Amazonbot | Amazon | Alexa / AI 검색 |
| Bytespider | ByteDance | TikTok AI |
| CCBot | Common Crawl | AI 학습 데이터셋 (오픈소스) |
| Applebot-Extended | Apple | Apple Intelligence |

robots.txt에서 이 User-Agent를 차단하면 해당 AI 플랫폼이 사이트를 크롤링하지 못한다.

In [ ]:
import urllib.request
import urllib.error
import re
import ssl
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from typing import Dict, List, Optional, Tuple

# SSL 검증 무시 (Colab 환경에서 일부 도메인 접속 시 필요)
ssl_ctx = ssl.create_default_context()
ssl_ctx.check_hostname = False
ssl_ctx.verify_mode = ssl.CERT_NONE

AI_CRAWLERS = {
    'GPTBot':           'OpenAI (ChatGPT 학습)',
    'ChatGPT-User':     'OpenAI (ChatGPT 브라우징)',
    'OAI-SearchBot':    'OpenAI (SearchGPT)',
    'Google-Extended':  'Google (Gemini AI)',
    'PerplexityBot':    'Perplexity AI',
    'ClaudeBot':        'Anthropic (Claude)',
    'Amazonbot':        'Amazon (Alexa AI)',
    'Bytespider':       'ByteDance (TikTok)',
    'CCBot':            'Common Crawl',
    'Applebot-Extended':'Apple Intelligence',
}

print(f'{len(AI_CRAWLERS)}개 AI 크롤러 정의 완료')

## 2. robots.txt 파서 

도메인의 robots.txt를 가져와서 AI 크롤러별 허용/차단 상태를 판별한다.

- **ALLOWED** — 명시적 허용 또는 언급 없음 (기본 허용)
- **BLOCKED** — `Disallow: /`로 전체 차단
- **PARTIAL** — 특정 경로만 차단 (예: `/api/`, `/admin/`)
- **WILDCARD_BLOCK** — `User-agent: *`에서 전체 차단 (모든 봇 차단)

In [ ]:
def fetch_robots_txt(domain: str) -> Optional[str]:
    """도메인의 robots.txt를 가져온다."""
    for scheme in ['https', 'http']:
        url = f'{scheme}://{domain}/robots.txt'
        try:
            req = urllib.request.Request(
                url,
                headers={'User-Agent': 'Mozilla/5.0 (compatible; GEO-Analyzer/1.0)'}
            )
            with urllib.request.urlopen(req, timeout=10, context=ssl_ctx) as resp:
                if resp.status == 200:
                    return resp.read().decode('utf-8', errors='replace')
        except Exception:
            continue
    return None


def parse_robots_rules(robots_text: str) -> Dict[str, List[Tuple[str, str]]]:
    """
    robots.txt를 파싱해서 User-agent별 규칙 목록을 반환한다.
    반환: { 'GPTBot': [('disallow', '/'), ...], '*': [('allow', '/'), ...] }
    """
    rules = {}
    current_agents = []

    for line in robots_text.splitlines():
        line = line.strip()
        if not line or line.startswith('#'):
            continue

        # User-agent 라인
        ua_match = re.match(r'^User-agent:\s*(.+)$', line, re.IGNORECASE)
        if ua_match:
            agent = ua_match.group(1).strip()
            current_agents.append(agent)
            if agent not in rules:
                rules[agent] = []
            continue

        # Allow/Disallow 라인
        rule_match = re.match(r'^(Allow|Disallow):\s*(.*)$', line, re.IGNORECASE)
        if rule_match and current_agents:
            directive = rule_match.group(1).lower()
            path = rule_match.group(2).strip()
            for agent in current_agents:
                rules[agent].append((directive, path))
            continue

        # 다른 디렉티브(Sitemap 등)를 만나면 current_agents 초기화
        if ':' in line and not line.lower().startswith(('allow', 'disallow')):
            current_agents = []

    return rules


def check_bot_status(rules: Dict, bot_name: str) -> str:
    """
    특정 봇의 차단/허용 상태를 판별한다.
    반환: 'ALLOWED', 'BLOCKED', 'PARTIAL', 'WILDCARD_BLOCK'
    """
    # 1. 봇 이름으로 직접 언급된 규칙 확인
    bot_rules = rules.get(bot_name, [])

    if bot_rules:
        disallows = [r for r in bot_rules if r[0] == 'disallow']
        allows = [r for r in bot_rules if r[0] == 'allow']

        # 전체 차단: Disallow: /
        if any(path == '/' for _, path in disallows):
            # Allow가 있으면 부분 허용 (Disallow: / + Allow: /specific)
            if allows:
                return 'PARTIAL'
            return 'BLOCKED'

        # 부분 차단
        if disallows and any(path for _, path in disallows):
            return 'PARTIAL'

        # 명시적 허용 또는 빈 Disallow
        return 'ALLOWED'

    # 2. 와일드카드(*) 규칙 확인
    wildcard_rules = rules.get('*', [])
    if wildcard_rules:
        disallows = [r for r in wildcard_rules if r[0] == 'disallow']
        if any(path == '/' for _, path in disallows):
            return 'WILDCARD_BLOCK'

    # 3. 언급 없음 = 기본 허용
    return 'ALLOWED'


def analyze_domain(domain: str) -> Dict[str, str]:
    """도메인 하나를 분석해서 AI 크롤러별 상태를 반환한다."""
    robots_text = fetch_robots_txt(domain)

    if robots_text is None:
        return {bot: 'NO_ROBOTS' for bot in AI_CRAWLERS}

    rules = parse_robots_rules(robots_text)
    return {bot: check_bot_status(rules, bot) for bot in AI_CRAWLERS}


print('파서 함수 준비 완료')

## 3. 도메인 분석 실행

한국 주요 기업 도메인을 대상으로 분석한다.

업종별로 섹션을 나눠두었다. 원하는 도메인으로 바꿔서 실행해보자.

In [ ]:
# ====================================================
# 이 리스트를 원하는 도메인으로 바꿔서 실행해보세요
# ====================================================

DOMAINS = {
    # 유통/커머스
    '커머스': [
        'coupang.com',
        'ssg.com',
        'musinsa.com',
        'oliveyoung.co.kr',
    ],
    # 플랫폼/테크
    '플랫폼': [
        'naver.com',
        'kakao.com',
        'toss.im',
        'baemin.com',
    ],
    # 여행/호텔
    '여행': [
        'yanolja.com',
        'goodchoice.kr',
    ],
    # 글로벌 비교군
    '글로벌': [
        'amazon.com',
        'nytimes.com',
        'reddit.com',
        'wikipedia.org',
    ],
}

# 전체 도메인 리스트 (flat)
all_domains = [d for group in DOMAINS.values() for d in group]

print(f'분석 대상: {len(all_domains)}개 도메인')
print('\n'.join(f'  - {d}' for d in all_domains))

In [ ]:
# 전체 도메인 분석 실행
results = {}

for domain in all_domains:
    print(f'분석 중: {domain}', end=' ... ')
    results[domain] = analyze_domain(domain)
    # 차단된 봇 수 카운트
    blocked = sum(1 for s in results[domain].values() if s in ('BLOCKED', 'WILDCARD_BLOCK'))
    print(f'{blocked}/{len(AI_CRAWLERS)} 차단')

print(f'\n분석 완료: {len(results)}개 도메인')

## 4. 결과 테이블

도메인 × AI 크롤러 매트릭스를 테이블로 확인한다.

In [ ]:
# 결과를 DataFrame으로 변환
STATUS_LABELS = {
    'ALLOWED':        '✅ 허용',
    'BLOCKED':        '❌ 차단',
    'PARTIAL':        '⚠️ 부분',
    'WILDCARD_BLOCK': '🚫 전체차단',
    'NO_ROBOTS':      '❓ 없음',
}

# 크롤러 이름을 짧게
short_names = {bot: desc.split('(')[0].strip() if '(' in desc else desc
               for bot, desc in AI_CRAWLERS.items()}

df = pd.DataFrame(
    {short_names[bot]: [STATUS_LABELS.get(results[d].get(bot, 'ALLOWED'), '?')
                        for d in all_domains]
     for bot in AI_CRAWLERS},
    index=all_domains
)
df.index.name = '도메인'

# 요약 통계
print('=== AI 크롤러 허용/차단 매트릭스 ===')
print()
df

## 5. 히트맵 시각화

차단 현황을 한눈에 볼 수 있는 히트맵으로 그린다.

- 파란색 = 허용, 빨간색 = 차단, 노란색 = 부분 차단, 회색 = robots.txt 없음

In [ ]:
# 상태를 숫자로 변환 (시각화용)
STATUS_NUMERIC = {
    'ALLOWED': 0,
    'PARTIAL': 1,
    'BLOCKED': 2,
    'WILDCARD_BLOCK': 3,
    'NO_ROBOTS': -1,
}

heatmap_data = pd.DataFrame(
    {short_names[bot]: [STATUS_NUMERIC.get(results[d].get(bot, 'ALLOWED'), -1)
                        for d in all_domains]
     for bot in AI_CRAWLERS},
    index=all_domains
)

# 커스텀 컀러맵
colors = ['#cccccc', '#4dabf7', '#ffd43b', '#ff6b6b', '#c92a2a']
bounds = [-1.5, -0.5, 0.5, 1.5, 2.5, 3.5]
cmap = mcolors.ListedColormap(colors)
norm = mcolors.BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(14, max(6, len(all_domains) * 0.6)))

im = ax.imshow(heatmap_data.values, cmap=cmap, norm=norm, aspect='auto')

# 라벨
ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns, rotation=45, ha='right', fontsize=10)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index, fontsize=10)

# 업종별 구분선
cumulative = 0
for sector, domains in DOMAINS.items():
    if cumulative > 0:
        ax.axhline(y=cumulative - 0.5, color='white', linewidth=2)
    mid = cumulative + len(domains) / 2 - 0.5
    ax.text(-1.8, mid, sector, ha='center', va='center',
            fontsize=9, fontweight='bold', color='#495057')
    cumulative += len(domains)

# 각 셀에 상태 텍스트
status_text = {0: 'O', 1: '△', 2: 'X', 3: 'X', -1: '?'}
for i in range(len(all_domains)):
    for j in range(len(AI_CRAWLERS)):
        val = heatmap_data.values[i, j]
        text_color = 'white' if val >= 2 else '#333'
        ax.text(j, i, status_text.get(val, '?'),
                ha='center', va='center', fontsize=11,
                fontweight='bold', color=text_color)

ax.set_title('AI 크롤러 허용/차단 히트맵', fontsize=14, fontweight='bold', pad=15)

# 범례
legend_labels = ['robots.txt 없음', '허용', '부분 차단', '차단', '전체 차단']
legend_handles = [plt.Rectangle((0,0),1,1, facecolor=c) for c in colors]
ax.legend(legend_handles, legend_labels, loc='upper center',
          bbox_to_anchor=(0.5, -0.12), ncol=5, fontsize=9)

plt.tight_layout()
plt.show()

## 6. 통계 요약

크롤러별 차단률과 도메인별 차단 크롤러 수를 정리한다.

In [ ]:
# 크롤러별 차단률
print('=== 크롤러별 차단률 ===')
print()
for bot in AI_CRAWLERS:
    statuses = [results[d][bot] for d in all_domains]
    blocked = sum(1 for s in statuses if s in ('BLOCKED', 'WILDCARD_BLOCK'))
    partial = sum(1 for s in statuses if s == 'PARTIAL')
    rate = blocked / len(statuses) * 100
    print(f'  {short_names[bot]:20s} 차단 {blocked:2d}/{len(statuses)}  '
          f'({rate:4.0f}%)  부분차단 {partial}건')

print()
print('=== 도메인별 AI 차단 크롤러 수 ===')
print()
for domain in all_domains:
    statuses = results[domain]
    blocked = sum(1 for s in statuses.values() if s in ('BLOCKED', 'WILDCARD_BLOCK'))
    partial = sum(1 for s in statuses.values() if s == 'PARTIAL')
    openness = '🟢 열림' if blocked == 0 else ('🟡 부분' if blocked < 5 else '🔴 차단')
    print(f'  {domain:25s} 차단 {blocked:2d}/{len(AI_CRAWLERS)}  '
          f'부분 {partial}  {openness}')

## 7. 내 도메인 직접 분석하기

아래 셀의 `MY_DOMAINS` 리스트를 수정해서 실행하면 된다.

In [ ]:
# ====================================================
# 여기에 분석할 도메인을 넣으세요
# ====================================================
MY_DOMAINS = [
    'example.com',       # 자사 도메인
    'competitor1.com',   # 경쟁사 1
    'competitor2.com',   # 경쟁사 2
]

my_results = {}
for domain in MY_DOMAINS:
    print(f'분석 중: {domain}', end=' ... ')
    my_results[domain] = analyze_domain(domain)
    blocked = sum(1 for s in my_results[domain].values()
                  if s in ('BLOCKED', 'WILDCARD_BLOCK'))
    print(f'{blocked}/{len(AI_CRAWLERS)} 차단')

# 결과 테이블
my_df = pd.DataFrame(
    {short_names[bot]: [STATUS_LABELS.get(my_results[d].get(bot, 'ALLOWED'), '?')
                        for d in MY_DOMAINS]
     for bot in AI_CRAWLERS},
    index=MY_DOMAINS
)
my_df.index.name = '도메인'
my_df

## 8. robots.txt 원문 확인

특정 도메인의 robots.txt 원문을 직접 확인하고 싶을 때.

In [ ]:
# 확인할 도메인을 수정하세요
TARGET = 'naver.com'

raw = fetch_robots_txt(TARGET)
if raw:
    # AI 크롤러 관련 라인만 하이라이트
    ai_bot_names = set(AI_CRAWLERS.keys())
    lines = raw.splitlines()
    print(f'=== {TARGET}/robots.txt ({len(lines)}행) ===')
    print()

    in_ai_section = False
    for line in lines:
        is_ai_line = any(bot.lower() in line.lower() for bot in ai_bot_names)
        if is_ai_line:
            in_ai_section = True
            print(f'\033[91m>>> {line}\033[0m')  # 빨간색 하이라이트
        elif in_ai_section and line.strip().lower().startswith(('allow', 'disallow')):
            print(f'\033[91m    {line}\033[0m')
        else:
            in_ai_section = False
            print(f'    {line}')
else:
    print(f'{TARGET}: robots.txt를 찾을 수 없습니다')

## 해석 가이드

| 상태 | 의미 | GEO 시사점 |
|------|------|----------|
| ✅ 허용 | AI 크롤러가 사이트를 자유롭게 크롤링 | On-Site GEO(JSON-LD, Schema) 효과 극대화 가능 |
| ⚠️ 부분 차단 | 일부 경로만 차단 | 허용된 경로에 구조화 데이터 집중 배치 |
| ❌ 차단 | 해당 AI 플랫폼이 사이트를 크롤링하지 못함 | On-Site GEO 무효. Off-Site 채널로 보완 필요 |
| 🚫 전체차단 | `User-agent: *`로 모든 봇 차단 | 의도적 차단인지 설정 오류인지 확인 필요 |
| ❓ 없음 | robots.txt 자체가 없음 | 모든 크롤러 접근 가능 (기본 허용) |

### 핵심 포인트

1. **GPTBot을 차단하면** ChatGPT 학습 데이터에서 제외된다. 단, ChatGPT-User(브라우징)는 별도 User-Agent라 브라우징 모드에서는 여전히 접근 가능할 수 있다.
2. **Google-Extended를 차단해도** 기본 Googlebot은 영향 없다. 검색 노출은 유지하면서 Gemini AI 학습만 차단하는 전략이 가능하다.
3. **경쟁사가 AI 크롤러를 차단하고 있다면**, 해당 AI 플랫폼에서 우리가 인용될 확률이 상대적으로 높아진다.